In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path
import multiprocessing as mp

# 1. Path Resolution
# Add the parent directory to the system path to import modules
# sys.path.append(str(Path(__file__).absolute().parent))
if 'haman' in os.getcwd():
    sys.path.append('/home/haman/mf-temporary/MeanFieldTester')
    sys.path.append('/home/haman/mf-temporary/MeanFieldTester/codes')
    repo_path = Path('/home/haman/mf-temporary')
elif 'pavel' in os.getcwd():
    sys.path.append('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester')
    sys.path.append('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester/codes')
    repo_path = Path('/home/pavel/academia/wintermute/mf-temporary')

# 2. The Great MPI Blindfold
keys_to_delete = [k for k in os.environ if k.startswith('SLURM') or k.startswith('OMPI_') or k.startswith('PRTE_') or 'HYDRA' in k]
for key in keys_to_delete:
    del os.environ[key]

# Prevent CPU hogging by the C++ backend
os.environ["OMP_NUM_THREADS"] = "1" 

# 3. Safe Multiprocessing
try:
    mp.set_start_method('spawn', force=True)
    print("Multiprocessing context set to:", mp.get_start_method())
except RuntimeError:
    pass

print(f"Loaded paths securely. MPI blindfolded. Ready for testing!")

In [ ]:
from importlib import reload

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from codes.neuron_simulation.config import NeuronSimulationConfig
from codes.transfer_function.config import TransferFunctionConfig
from codes.network_params.loader import load_network_parameters
from codes.data_structures.single_neuron import SingleNeuronResults
from codes.network_params.models import BiologicalParameters

from codes.plotting import fig_plots
import codes.neuron_simulation as ns
import codes.transfer_function as tf


from pydantic import BaseModel

project_path = repo_path / "projects" / "01_fitting_tf"
os.chdir(project_path)

class TestNeuronSimulationParams(BaseModel):
    neuron_simulation: NeuronSimulationConfig

class TestTransferFunctionParams(BaseModel):
    transfer_function: TransferFunctionConfig

In [ ]:
network_params = load_network_parameters(project_path / "params" / "divolo2019_network.yaml")
neuron_simulators = ["zerlaut2018", "pynn.nest"]
transfer_functions = ["zerlaut2018", "divolo2019", "neuropsi.custom"]



# Part I:
Here I directly generate neuron data based on cell properties in the paper.
Then I make fit myself and compare it with the paper fit.

In [ ]:
# generate test data for transfer function workflow

test_workflow_params = {
    "neuron_simulation": {
        "execution_mode": "run",
        "simulator": "pynn.nest",
        "grid": {
            "exc_neuron" : {
                "grid_type": "adaptive",
                "exc_rate_grid": "adaptive",
                "inh_rate_grid": [0, 30, 16],
                "out_rate_grid": [0, 30, 16],
            },
            "inh_neuron" : {
                "grid_type": "adaptive",
                "exc_rate_grid": "adaptive",
                "inh_rate_grid": [0, 30, 16],
                "out_rate_grid": [0, 30, 16],
            }
        },
        "simulation_time": 15000.0,
        "averaging_window": 10000.0,
        "time_step": 0.1,
        "seed": 42,
        "n_runs": 5,
        "cpus": 16
    }
}

test_config = TestNeuronSimulationParams(**test_workflow_params).neuron_simulation

print("Config loaded successfully!")
print(f"Running for {test_config.simulation_time} ms")

neuron_results = ns.run_neuron_simulation_workflow(test_config, network_params)


In [ ]:
fig_name = f"neuron_activity_dummy_data_adaptive_grid_{test_config.simulator.replace('.', '_')}.png"
fig_path = project_path / "imgs" / fig_name
fig_plots.fig_neuron_activity(
    neuron_results, 
    common_params={}, 
    fig_params={
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
        'title': f"Neuron Activity - {test_config.simulator}"
    }
)
display(Image(filename=str(fig_path)))

fig_name = f"std_neuron_activity_dummy_data_adaptive_grid_{test_config.simulator.replace('.', '_')}.png"
fig_path = project_path / "imgs" / fig_name
fig_plots.fig_tf_fits_together(
    neuron_results, 
    {"exc_neuron": [],
    "inh_neuron": []
    }, 
    common_params={
        'labels' : [],
        'linestyles' : [],
        # 'xlim' : (0,7),
        'ylim' : (0, 30),
    }, 
    fig_params={
        'fontsize': 14,
        'figsize': (15, 10),  # width, height
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
        'title': f"Neuron Activity - {test_config.simulator} (STD)"
    })
display(Image(filename=str(fig_path)))




In [ ]:
divolo_tf_params = {
    "transfer_function": {
        "fit_transfer_function": False,
        "tf_model": {
            "model_name": "divolo2019",
            "square_terms": True,
            "log_term": False,
        },
        "expansion_point": {
            "voltage_mean": -60.0,
            "voltage_std": 4.0,
            "voltage_tau": 0.5
        },
        "expansion_norm": {
            "voltage_mean": 10.0,
            "voltage_std": 6.0,
            "voltage_tau": 1.0
        },
        "tf_fits": None,
    }
}


paper_fit_params = {
    "exc_neuron" : {
        # Di Volo 2019 paper
        # "exc_neuron" : [-49.8, 5.06, -25.0, 1.4, -0.41, 10.5, -36.0, 7.4, 1.2, -40.7],
        "P_0": -49.8,
        "P_mean": 5.06,
        "P_std": -25.0,
        "P_tau": 1.4,
        "P_log": 0.0,
        "P_mean_mean": -0.41,
        "P_mean_std": 7.4,
        "P_mean_tau": 1.2,
        "P_std_std": 10.5,
        "P_std_tau": -40.7,
        "P_tau_tau": -36.0,
    },
    "inh_neuron" : {
        # Di Volo 2019 paper
        # "inh_neuron" : [-51.4, 4.0, -8.3, 0.2, -0.5, 1.4, -14.6, 4.5, 2.8, -15.3],    
        "P_0": -51.4,
        "P_mean": 4.0,
        "P_std": -8.3,
        "P_tau": 0.2,
        "P_log": 0.0,
        "P_mean_mean": -0.5,
        "P_mean_std": 4.5,
        "P_mean_tau": 2.8,
        "P_std_std": 1.4,
        "P_std_tau": -15.3,
        "P_tau_tau": -14.6,
    }
}

repo_fit_params = {
    "exc_neuron": {
        # Di Volo repo
        # "exc_neuron" : [-49.83106, 5.06355, -23.47012, 2.29515, -0.41053, 10.54705, -36.59253, 7.43749, 1.26506, -40.72161],
        "P_0": -49.83106,
        "P_mean": 5.06355,
        "P_std": -23.47012,
        "P_tau": 2.29515,
        "P_log": 0.0,
        "P_mean_mean": -0.41053,
        "P_mean_std": 7.43749,
        "P_mean_tau": 1.26506,
        "P_std_std": 10.54705,
        "P_std_tau": -40.72161,
        "P_tau_tau": -36.59253
    },
    "inh_neuron": {
        # Di Volo repo
        # "inh_neuron" : [-51.49122, 4.00369, -8.35201, 0.24142, -0.50706, 1.43454, -14.68669, 4.50271, 2.84722, -15.3578],    
        "P_0": -51.49122,
        "P_mean": 4.00369,
        "P_std": -8.35201,
        "P_tau": 0.24142,
        "P_log": 0.0,
        "P_mean_mean": -0.50706,
        "P_mean_std": 4.50271,
        "P_mean_tau": 2.84722,
        "P_std_std": 1.43454,
        "P_std_tau": -15.3578,
        "P_tau_tau": -14.68669
    }
}


custom_tf_params = {
    "transfer_function": {
        "fit_transfer_function": True,
        "tf_model": {
            "model_name": "neuropsi.custom",
            "square_terms": True,
            "log_term": False,
            "adaptation": True
        },
        "expansion_point": {
            "voltage_mean": -60.0,
            "voltage_std": 4.0,
            "voltage_tau": 0.5
        },
        "expansion_norm": {
            "voltage_mean": 10.0,
            "voltage_std": 6.0,
            "voltage_tau": 1.0
        },
        "fit_with_w": True,
        "out_rate_min": 0.0,
        "out_rate_max": 200.0,
        "V_eff_fitting": {
            "method": "SLSQP",
            "options": {
                "ftol": 5e-9,
                "disp": False,
                "maxiter": 10000
            }
        },
        "TF_fitting": {
            "method": "nelder-mead",
            "options": {
                "fatol": 5e-9,
                "disp": False,
                "maxiter": 10000
            }
        }
    }
}

In [ ]:
divolo_tf_params["transfer_function"]["tf_fits"] = paper_fit_params
# divolo_tf_params["transfer_function"]["tf_fits"] = repo_fit_params

tf_divolo2019_params = TestTransferFunctionParams(**divolo_tf_params).transfer_function
tf_custom_params = TestTransferFunctionParams(**custom_tf_params).transfer_function


tf_funcs_legacy = {
    "exc_neuron": [],
    "inh_neuron": []
}
for tf_params in [tf_divolo2019_params, tf_custom_params]:
    tf_funcs = tf.run_tf_fitting_workflow(tf_params, network_params, neuron_results)
    for neuron_type in tf_funcs.keys():
        tf_funcs_legacy[neuron_type].append(tf_funcs[neuron_type])


In [ ]:
fig_name = f"std_neuron_activity_dummy_data_linear_grid_{test_config.simulator.replace('.', '_')}.png"
fig_path = project_path / "imgs" / fig_name
fig_plots.fig_tf_fits_together(
    neuron_results, 
    tf_funcs_legacy, 
    common_params={
        'labels' : ["DiVolo paper", "MFT new fit"],
        'linestyles' : ['--', ":"],
        # 'xlim' : (0,7),
        'ylim' : (0, 30),
    }, 
    fig_params={
        'fontsize': 14,
        'figsize': (15, 10),  # width, height
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
        'title': f"Neuron Activity - {test_config.simulator} (STD)"
    })
display(Image(filename=str(fig_path)))

DiVolo publish (and repo) fit is different...
We will be investigating now, why

# Part II - inspecting Zerlaut2018_data and data generation

Apparently there is something odd

1. Use Zerlaut2018_simulator to generate data
1. compare data from simulator and from the script
1. pynn.nest, zerlaut simulator data





### Zerlaut2018 data

In [ ]:
zerlaut_repo_results = {}
for neuron_name, cell_type in zip(network_params.internal_neurons, ["RS-cell", "FS-cell"]):
    zerlaut_data = np.load(project_path / "zerlaut2018_data" / f"{cell_type}_CONFIG1.npy", allow_pickle=True)
    
    zerlaut_repo_results[neuron_name] = SingleNeuronResults(
        neuron_name=cell_type,
        neuron_params=zerlaut_data[4],

        exc_rate_grid=zerlaut_data[2].T,
        inh_rate_grid=np.stack([zerlaut_data[3]]*10, axis=1).T,
        out_rate_mean=zerlaut_data[0].T,
        out_rate_std=zerlaut_data[1].T)


fig_name = "Zerlaut_data_neuron_activity.png"
fig_path = project_path / "imgs" / fig_name
fig_plots.fig_neuron_activity(
    zerlaut_repo_results, 
    common_params={}, 
    fig_params={
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
        'title': f"Neuron Activity - Zerlaut2018 repo data"
    }
)
display(Image(filename=str(fig_path)))

In [ ]:
zerlaut_network_params = {
    "neurons":{
        "exc_neuron": {
            "neuron_type" : "excitatory",
            "neuron_model" : "adex",
            "neuron_params": {
                "v_rest": zerlaut_repo_results["exc_neuron"].neuron_params["El"]*1e3,
                "v_reset": zerlaut_repo_results["exc_neuron"].neuron_params["Vreset"]*1e3,
                "tau_refrac": zerlaut_repo_results["exc_neuron"].neuron_params["Trefrac"]*1e3,
                "tau_m": zerlaut_repo_results["exc_neuron"].neuron_params["Cm"]/zerlaut_repo_results["exc_neuron"].neuron_params["Gl"]*1e3,
                "cm": zerlaut_repo_results["exc_neuron"].neuron_params["Cm"]*1e9,
                "e_rev_E": zerlaut_repo_results["exc_neuron"].neuron_params["Ee"]*1e3,
                "e_rev_I": zerlaut_repo_results["exc_neuron"].neuron_params["Ei"]*1e3,
                "tau_syn_E": zerlaut_repo_results["exc_neuron"].neuron_params["Te"]*1e3,
                "tau_syn_I": zerlaut_repo_results["exc_neuron"].neuron_params["Ti"]*1e3,
                "a": zerlaut_repo_results["exc_neuron"].neuron_params["a"]*1e9,
                "b": zerlaut_repo_results["exc_neuron"].neuron_params["b"]*1e9,
                "delta_T": zerlaut_repo_results["exc_neuron"].neuron_params["delta_v"]*1e3,
                "tau_w": zerlaut_repo_results["exc_neuron"].neuron_params["tauw"]*1e3,
                "v_thresh": zerlaut_repo_results["exc_neuron"].neuron_params["Vthre"]*1e3,
            }
        },
        "inh_neuron": {
            "neuron_type" : "inhibitory",
            "neuron_model" : "adex",
            "neuron_params": {
                "v_rest": zerlaut_repo_results["inh_neuron"].neuron_params["El"]*1e3,
                "v_reset": zerlaut_repo_results["inh_neuron"].neuron_params["Vreset"]*1e3,
                "tau_refrac": zerlaut_repo_results["inh_neuron"].neuron_params["Trefrac"]*1e3,
                "tau_m": zerlaut_repo_results["inh_neuron"].neuron_params["Cm"]/zerlaut_repo_results["inh_neuron"].neuron_params["Gl"]*1e3,
                "cm": zerlaut_repo_results["inh_neuron"].neuron_params["Cm"]*1e9,
                "e_rev_E": zerlaut_repo_results["inh_neuron"].neuron_params["Ee"]*1e3,
                "e_rev_I": zerlaut_repo_results["inh_neuron"].neuron_params["Ei"]*1e3,
                "tau_syn_E": zerlaut_repo_results["inh_neuron"].neuron_params["Te"]*1e3,
                "tau_syn_I": zerlaut_repo_results["inh_neuron"].neuron_params["Ti"]*1e3,
                "a": zerlaut_repo_results["inh_neuron"].neuron_params["a"]*1e9,
                "b": zerlaut_repo_results["inh_neuron"].neuron_params["b"]*1e9,
                "delta_T": zerlaut_repo_results["inh_neuron"].neuron_params["delta_v"]*1e3,
                "tau_w": zerlaut_repo_results["inh_neuron"].neuron_params["tauw"]*1e3,
                "v_thresh": zerlaut_repo_results["inh_neuron"].neuron_params["Vthre"]*1e3,
            },
        },
        "drive_neuron": {
            "neuron_type" : "excitatory",
            "neuron_model" : "poisson_generator",
        },
        "stim_neuron": {
            "neuron_type" : "excitatory",
            "neuron_model" : "poisson_generator",
        }
    },
    "network": {
        "size": {
            "exc_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["Ntot"]*(1-zerlaut_repo_results["exc_neuron"].neuron_params["gei"]),
            "inh_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["Ntot"]*zerlaut_repo_results["exc_neuron"].neuron_params["gei"],
            "drive_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["Ntot"]*(1-zerlaut_repo_results["exc_neuron"].neuron_params["gei"]),
            "stim_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["Ntot"]*(1-zerlaut_repo_results["exc_neuron"].neuron_params["gei"]),
        },
        "connectivity": {
            "exc_neuron":{
                "exc_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["pconnec"],
                "inh_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["pconnec"],
                "drive_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["pconnec"],
                "stim_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["pconnec"]
            },
            "inh_neuron": {
                "exc_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["pconnec"],
                "inh_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["pconnec"],
                "drive_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["pconnec"],
                "stim_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["pconnec"],

            },
            "drive_neuron": {
                "exc_neuron": 0.0,
                "inh_neuron": 0.0,
                "drive_neuron": 0.0,
                "stim_neuron": 0.0,

            },
            "stim_neuron":{
                "exc_neuron": 0.0,
                "inh_neuron": 0.0,
                "drive_neuron": 0.0,
                "stim_neuron": 0.0,
            }
        }
    },
    "synapses": {
        "exc_neuron": {
            "syn_type": "static_synapse",
            "syn_params": {
                "weight": zerlaut_repo_results["exc_neuron"].neuron_params["Qe"]*1e9,
                "delay": 1.0
            }
        },
        "inh_neuron": {
            "syn_type": "static_synapse",
            "syn_params": {
                "weight": zerlaut_repo_results["exc_neuron"].neuron_params["Qi"]*1e9,
                "delay": 1.0
            }
        },
        "drive_neuron": {
            "syn_type": "static_synapse",
            "syn_params": {
                "weight": zerlaut_repo_results["exc_neuron"].neuron_params["Qe"]*1e9,
                "delay": 1.0
            }

        },
        "stim_neuron": {
            "syn_type": "static_synapse",
            "syn_params": {
                "weight": zerlaut_repo_results["exc_neuron"].neuron_params["Qe"]*1e9,
                "delay": 1.0
            }

        }
    }
}

zerlaut_network_params = BiologicalParameters(**zerlaut_network_params)

In [ ]:
zerlaut_workflow_params = {
    "neuron_simulation": {
        "execution_mode": "run",
        "simulator": "zerlaut2018",
        "grid": {
            "exc_neuron" : {
                "grid_type": "adaptive",
                "exc_rate_grid": "adaptive",
                "inh_rate_grid": [0, 30, 10],
                "out_rate_grid": [0, 30, 10],
            },
            "inh_neuron" : {
                "grid_type": "adaptive",
                "exc_rate_grid": "adaptive",
                "inh_rate_grid": [0, 30, 10],
                "out_rate_grid": [0, 30, 10],
            }
        },
        "simulation_time": 10000.0,
        "averaging_window": 10000.0,
        "time_step": 0.05,
        "seed": 4,
        "n_runs": 5,
        "cpus": 16
    }
}

zerlaut_workflow_params = {
    "neuron_simulation": {
        "execution_mode": "run",
        "simulator": "zerlaut2018",
        "grid": {
            "exc_neuron" : {
                "grid_type": "custom",
                "exc_rate_grid": zerlaut_repo_results["exc_neuron"].exc_rate_grid(),
                "inh_rate_grid": zerlaut_repo_results["exc_neuron"].inh_rate_grid(),
            },
            "inh_neuron" : {
                "grid_type": "custom",
                "exc_rate_grid": zerlaut_repo_results["inh_neuron"].exc_rate_grid(),
                "inh_rate_grid": zerlaut_repo_results["inh_neuron"].inh_rate_grid(),
            }
        },
        "simulation_time": 10000.0,
        "averaging_window": 10000.0,
        "time_step": 0.05,
        "seed": 1,
        "n_runs": 3,
        "cpus": 16
    }
}

zerlaut_config = TestNeuronSimulationParams(**zerlaut_workflow_params).neuron_simulation

print("Config loaded successfully!")
print(f"Running for {zerlaut_config.simulation_time} ms")

zerlaut_mft_results = ns.run_neuron_simulation_workflow(zerlaut_config, zerlaut_network_params)

In [ ]:
pynn_workflow_params = {
    "neuron_simulation": {
        "execution_mode": "run",
        "simulator": "pynn.nest",
        "grid": {
            "exc_neuron" : {
                "grid_type": "custom",
                "exc_rate_grid": zerlaut_repo_results["exc_neuron"].exc_rate_grid(),
                "inh_rate_grid": zerlaut_repo_results["exc_neuron"].inh_rate_grid(),
            },
            "inh_neuron" : {
                "grid_type": "custom",
                "exc_rate_grid": zerlaut_repo_results["inh_neuron"].exc_rate_grid(),
                "inh_rate_grid": zerlaut_repo_results["inh_neuron"].inh_rate_grid(),
            }
        },
        "simulation_time": 10000.0,
        "averaging_window": 10000.0,
        "time_step": 0.05,
        "seed": 4,
        "n_runs": 5,
        "cpus": 16
    }
}

pynn_config = TestNeuronSimulationParams(**pynn_workflow_params).neuron_simulation

print("Config loaded successfully!")
print(f"Running for {pynn_config.simulation_time} ms")

pynn_mft_results = ns.run_neuron_simulation_workflow(pynn_config, zerlaut_network_params)

In [ ]:
neuron_results_list = [zerlaut_repo_results, zerlaut_mft_results, pynn_mft_results]
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

result_labels = ["Zerlaut repo data", "Zerlaut MFT data", "PyNN MFT data"]

for label, neuron_results, marker in zip(result_labels, neuron_results_list, ['o', 'x', '+']):
    axes[0].set_prop_cycle(None)
    axes[1].set_prop_cycle(None)    
    lines_exc = axes[0].plot(neuron_results['exc_neuron'].exc_rate_grid(), 
                 neuron_results['exc_neuron'].out_rate_mean(), 
                 ls='',
                 marker=marker)
    
    lines_inh = axes[1].plot(neuron_results['inh_neuron'].exc_rate_grid(), 
                 neuron_results['inh_neuron'].out_rate_mean(), 
                 ls='',
                 marker=marker)
    lines_exc[0].set_label(label)
    lines_inh[0].set_label(label)
axes[0].set_title("Excitatory Neuron")
axes[1].set_title("Inhibitory Neuron")
axes[0].set_xlabel("Excitatory Input Rate (Hz)")
axes[1].set_xlabel("Excitatory Input Rate (Hz)")
axes[0].set_ylabel("Output Rate (Hz)")
axes[1].set_ylabel("Output Rate (Hz)")
axes[0].legend()
axes[1].legend()
fig.suptitle("Comparison of Neuron Activity Across Simulators")
fig.tight_layout()

I do not know the params to run the data generation the same as the repo data were generated

All the generated data are close enough

# Part III - inspecting TF fitting

- NOTE: Zerlaut originally had a mistake in get_fluct_regime_vars function

- fix the data generated (MFT data, as adaptation is needed)

- It does not make sense to test the fitting results! It is important to check
that each implementation give the same TF curve results with same fit parameters.
Fitting is completely chaotic and may lead to different results easily

- Zerlaut repo fit, vs zerlaut script fit, vs zerlaut MFT fit vs neuropsi MFT fit
- Divolo repo fit, vs divolo script fit, vs divolo MFT fit vs neuropsi MFT fit


In [ ]:
import codes.transfer_function.zerlaut2018_tf as zerlaut_tf
import codes.transfer_function.neuropsi_tf as neuropsi_tf
import codes.transfer_function.divolo2019_tf as divolo_tf

In [ ]:
reload(tf)
reload(neuropsi_tf)

### Zerlaut (ignoring adaptation)

In [ ]:
zerlaut_mft_tf_params = {
    "transfer_function": {
        "fit_transfer_function": True,
        "tf_model": {
            "model_name": "zerlaut2018",
            "square_terms": True,
            "log_term": True,
        },
        "expansion_point": {
            "voltage_mean": -60.0,
            "voltage_std": 4.0,
            "voltage_tau": 0.5
        },
        "expansion_norm": {
            "voltage_mean": 10.0,
            "voltage_std": 6.0,
            "voltage_tau": 1.0
        },
        "fit_with_w": True,
        "out_rate_min": 0.0,
        "out_rate_max": 200.0,
        "V_eff_fitting": {
            "method": "SLSQP",
            "options": {
                "disp": False,
                "ftol": 1e-8,
                "maxiter": 40000,
            }
        },
        "TF_fitting": {
            "method": "nelder-mead",
            "options": {
                "xtol": 1e-5,
                "disp": False,
                "maxiter": 10000
            },
        }
    }
}

custom_mft_tf_params = {
    "transfer_function": {
        "fit_transfer_function": True,
        "tf_model": {
            "model_name": "neuropsi.custom",
            "square_terms": True,
            "log_term": True,
            "adaptation": False
        },
        "expansion_point": {
            "voltage_mean": -60.0,
            "voltage_std": 4.0,
            "voltage_tau": 0.5
        },
        "expansion_norm": {
            "voltage_mean": 10.0,
            "voltage_std": 6.0,
            "voltage_tau": 1.0
        },
        "fit_with_w": True,
        "out_rate_min": 0.0,
        "out_rate_max": 200.0,
        "V_eff_fitting": {
            "method": "SLSQP",
            "options": {
                "disp": False,
                'ftol': 1e-2,  # tolerance has to be 1e6 times higher because of units in Zerlaut and custom
                'maxiter':40000,
            }
        },
        "TF_fitting": {
            "method": "nelder-mead",
            "options": {
                "disp": False,
                "xtol": 1e-5,
                "maxiter": 10000, 
            },
        }
    }
}

zerlaut_mft_tf_params = TestTransferFunctionParams(**zerlaut_mft_tf_params).transfer_function
custom_mft_tf_params = TestTransferFunctionParams(**custom_mft_tf_params).transfer_function



In [ ]:
# zerlaut TF fit repo, zerlaut TF fit script, zerlaut TF fit MFT, neuropsi_tf MFT

# NOTE: When you want to prove two implementations work the exact same way
# regardless of units, you don't test the optimizer—you test the mathematical
# surface. Optimizers are chaotic algorithms that navigate the surface; 
# if we want to prove the models are identical, we just need to prove that they 
# calculate the exact same error (MSE) for a fixed set of parameters.

tf_fit_results = {}
for neuron_name, neuron_type in zip(["exc_neuron", "inh_neuron"], ["RS-cell", "FS-cell"]):
    zerlaut_repo_fit = np.load(project_path / "zerlaut2018_data" / f"{neuron_type}_CONFIG1_repo_fit.npy", allow_pickle=True)
    zerlaut_script_fit = zerlaut_tf.make_fit_from_data(f"./zerlaut2018_data/{neuron_type}_CONFIG1.npy", with_square_terms=True)
    tf_fit_results[neuron_name] = {
        "zerlaut_repo_fit": zerlaut_repo_fit*1e3,
        "zerlaut_script_fit": zerlaut_script_fit*1e3
    }

neuron_results = zerlaut_repo_results

zerlaut_mft_tf = tf.run_tf_fitting_workflow(zerlaut_mft_tf_params, zerlaut_network_params, neuron_results)
neuropsi_mft_tf = tf.run_tf_fitting_workflow(custom_mft_tf_params, zerlaut_network_params, neuron_results)

for neuron_name in ["exc_neuron", "inh_neuron"]:
    tf_fit_results[neuron_name]["zerlaut_mft_fit"] = np.array(list(zerlaut_mft_tf[neuron_name].fitted_params.values()))
    tf_fit_results[neuron_name]["neuropsi_mft_fit"] = np.array(list(neuropsi_mft_tf[neuron_name].fitted_params.values()))

In [ ]:
for neuron_name, results in tf_fit_results.items():
    print("\n")
    print(f"Results for {neuron_name}:")
    print(f"{'Parameter':11} | {'Repo':10} | {'Script':10} | {'Zer MFT':10} | {'Custom MFT':10}")
    print(f"{'-'*11} | {'-'*10} | {'-'*10} | {'-'*10} | {'-'*10}")
    for a,b,c,d, key in zip(*results.values(), list(neuropsi_mft_tf[neuron_name].fitted_params.keys())):
        print(f"{key:11} | {a:10.5f} | {b:10.5f} | {c:10.5f} | {d:10.5f} ")


In [ ]:
# Although different methods give different fitted parameters, they should all
# give the same MSE when we calculate the output rate with those parameters and
# compare to the original output rate from the neuron simulation. This is because
# the mathematical surface of the error should be the same for all implementations, 
# even if the optimizers navigate it differently and end up in different local minima.

for neuron_name, results in tf_fit_results.items():
    print("\n")
    print(f"Results for {neuron_name}:")
    print(f"{'Fit Source':18} | {'Script MSE':11} | {'Zer MFT MSE':11} | {'Custom MFT MSE':10}")
    print(f"{'-'*18} | {'-'*11} | {'-'*11} | {'-'*10}")

    inh_rate = neuron_results[neuron_name].inh_rate_grid()
    exc_rate = neuron_results[neuron_name].exc_rate_grid()
    out_rate = neuron_results[neuron_name].out_rate_mean()
    params = neuron_results[neuron_name].neuron_params
    Qe = params["Qe"]
    Te = params["Te"]
    Ee = params["Ee"]
    Qi = params["Qi"]
    Ti = params["Ti"]
    Ei = params["Ei"]
    Gl = params["Gl"]
    Cm = params["Cm"]
    El = params["El"]
    Ntot = params["Ntot"]
    pconnec = params["pconnec"]
    gei = params["gei"]

    for fit_source, fit_params in results.items():

        out_rate_script = zerlaut_tf.TF_my_template(exc_rate, inh_rate, Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, *(fit_params*1e-3))
        mse_script = np.mean((out_rate - out_rate_script)**2)

        fit_params = dict(zip(["P_0", "P_mean", "P_std", "P_tau", "P_log", "P_mean_mean", "P_std_std", "P_tau_tau", "P_mean_std", "P_mean_tau", "P_std_tau"], fit_params))

        divolo_mft = zerlaut_tf.Zerlaut2018TF(neuron_name, zerlaut_network_params, zerlaut_mft_tf_params)

        divolo_mft.set_fitted_parameters(fit_params)

        out_rate_divolo_mft = divolo_mft(exc_rate=exc_rate, inh_rate=inh_rate)
        mse_divolo_mft = np.mean((out_rate - out_rate_divolo_mft)**2)
        
        custom_mft = neuropsi_tf.NeuroPSICustomTF(neuron_name, zerlaut_network_params, custom_mft_tf_params)
        custom_mft.set_fitted_parameters(fit_params)
        out_rate_custom_mft = custom_mft(exc_rate=exc_rate, inh_rate=inh_rate)
        mse_custom_mft = np.mean((out_rate - out_rate_custom_mft)**2)
        print(f"{fit_source:18} | {mse_script:11.5f} | {mse_divolo_mft :11.5f} | {mse_custom_mft :10.5f}")

In [ ]:
# membrane fluctuation parameters
# Zerlaut script, di Volo script, MFT custom MembranePotentialFluctuations

step=1

fig, ax = plt.subplots(nrows=5, ncols=2, figsize=(16,16))
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
for i, neuron_name in enumerate(["exc_neuron", "inh_neuron"]):
    for inh_idx in range(neuron_results[neuron_name].inh_rate_grid().shape[1]):
        if inh_idx % step != 0:
            continue
        inh_rate = neuron_results[neuron_name].inh_rate_grid()[:, inh_idx]
        exc_rate = neuron_results[neuron_name].exc_rate_grid()[:, inh_idx]
        out_rate = neuron_results[neuron_name].out_rate_mean()[:, inh_idx]
        params = neuron_results[neuron_name].neuron_params
        Qe = params["Qe"]
        Te = params["Te"]
        Ee = params["Ee"]
        Qi = params["Qi"]
        Ti = params["Ti"]
        Ei = params["Ei"]
        Gl = params["Gl"]
        Cm = params["Cm"]
        El = params["El"]
        Ntot = params["Ntot"]
        pconnec = params["pconnec"]
        gei = params["gei"]

        # cycle through default matplotlib colors
        color = colors[inh_idx % len(colors)]

        zerlaut_mpf = zerlaut_tf.get_fluct_regime_vars(exc_rate, inh_rate, Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, *([0]*11))
        divolo_mpf= divolo_tf.get_fluct_regime_varsup(exc_rate, inh_rate, np.zeros_like(exc_rate), Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, *([0]*11))
        custom_mpf = neuropsi_tf.MembranePotentialFluctuations(neuron_name=neuron_name, network_params=zerlaut_network_params).evaluate(exc_rate, inh_rate)
        assert np.all(zerlaut_mpf[0]==divolo_mpf[0])
        assert np.all(zerlaut_mpf[1]==divolo_mpf[1])
        assert np.all(zerlaut_mpf[2]==divolo_mpf[2])
        assert np.all(zerlaut_mpf[3]==divolo_mpf[3])
        assert np.allclose(zerlaut_mpf[0]*1e3, custom_mpf[0], atol=1e-13, rtol=0)
        assert np.allclose(zerlaut_mpf[1]*1e3, custom_mpf[1], atol=1e-14, rtol=0)
        assert np.allclose(zerlaut_mpf[2], custom_mpf[4]/zerlaut_network_params.neurons[neuron_name].neuron_params.g_L, atol=1e-14, rtol=0)
        assert np.allclose(zerlaut_mpf[3], custom_mpf[3], atol=1e-15, rtol=0)


        ax[0,i].plot(exc_rate, zerlaut_mpf[0]*1e3, 'x', color=color, label=fr"$\nu_{{inh}}={inh_rate[0]:.1f} Hz$")
        ax[0,i].plot(exc_rate, divolo_mpf[0]*1e3, 'o', color=color)
        ax[0,i].plot(exc_rate, custom_mpf[0], '+', color=color)
        # print(f"Total difference in muV for inh={inh_input:.1f} Hz : ", np.mean(np.abs(muV*1e3 - mu_V)), "mV")

        ax[1,i].plot(exc_rate, zerlaut_mpf[1]*1e3, 'x', color=color, label=fr"$\nu_{{inh}}={inh_rate[0]:.1f} Hz$")
        ax[1,i].plot(exc_rate, divolo_mpf[1]*1e3, 'o', color=color)
        ax[1,i].plot(exc_rate, custom_mpf[1], '+', color=color)
        # print(f"Total difference in sigmaV for inh={inh_input:.1f} Hz : ", np.mean(np.abs(sV*1e3 - sigma_V)), "mV")

        ax[2,i].plot(exc_rate, zerlaut_mpf[3], 'x', color=color, label=fr"$\nu_{{inh}}={inh_rate[0]:.1f} Hz$")
        ax[2,i].plot(exc_rate, divolo_mpf[3], 'o', color=color)
        ax[2,i].plot(exc_rate, custom_mpf[3], '+', color=color)
        # print(f"Total difference in tauVN for inh={inh_input:.1f} Hz : ", np.mean(np.abs(TvN - tau_VN)), "ms")
        
        ax[3,i].plot(exc_rate, zerlaut_mpf[2], 'x', color=color, label=fr"$\nu_{{inh}}={inh_rate[0]:.1f} Hz$")
        ax[3,i].plot(exc_rate, divolo_mpf[2], 'o', color=color)
        ax[3,i].plot(exc_rate, custom_mpf[4]/zerlaut_network_params.neurons[neuron_name].neuron_params.g_L, '+', color=color)
        # print(f"Total difference in muG for inh={inh_input:.1f} Hz : ", np.mean(np.abs(muGn*1e9 - mu_G/Gl)), "nS")

        mask = out_rate > 0

        zerlaut_v_eff = zerlaut_tf.effective_Vthre(out_rate, zerlaut_mpf[0], zerlaut_mpf[1], zerlaut_mpf[3], Gl, Cm)
        divolo_v_eff = divolo_tf.effective_Vthre(out_rate, divolo_mpf[0], divolo_mpf[1], divolo_mpf[3], Gl, Cm)

        # TODO: change to tf results from above
        custom_tf = neuropsi_tf.NeuroPSICustomTF(neuron_name, zerlaut_network_params, custom_mft_tf_params)
        custom_v_eff = custom_tf._get_target_v_eff(out_rate, *custom_mpf[:3])

        assert np.all(zerlaut_v_eff==divolo_v_eff)
        assert np.allclose(zerlaut_v_eff[mask]*1e3, custom_v_eff[mask], atol=1e-13, rtol=0)

        ax[4,i].plot(exc_rate[mask], zerlaut_v_eff[mask]*1e3, 'x', color=color, label=fr"$\nu_{{inh}}={inh_rate[0]:.1f} Hz$")
        ax[4,i].plot(exc_rate[mask], divolo_v_eff[mask]*1e3, 'o', color=color)
        ax[4,i].plot(exc_rate[mask], custom_v_eff[mask], '+', color=color)
        # print(f"Total difference in Vthre_eff for inh={inh_input:.1f} Hz : ", np.mean(np.abs(Vthre_eff*1e3 - v_eff)), "mV")

    ax[0,i].set_title(f"{neuron_name.replace('_', ' ').capitalize()}")
    ax[0,i].set_ylabel(r"$\mu_V$ (mV)")
    ax[1,i].set_ylabel(r"$\sigma_V$ (mV)")
    ax[2,i].set_ylabel(r"$\tau_V^N$ (ms)")
    ax[3,i].set_ylabel(r"$\mu_{G}/G_l$ (nS)")
    ax[4,i].set_ylabel(r"$V_{thre}^{eff}$ (mV)")
    ax[-1,i].set_xlabel(r"$\nu_{exc}$ (Hz)")
fig.tight_layout()
# plt.legend()

- Data generated are close enough regardless of the method
- Membrane potential fluctuation parameters are close enough regardless od the dealing with zero values
- Fitting gives different results in second part (nelder-mead), where the units play role as the step is of fixed value and does not rescale
- All TF implementations give the same SME provided with the same TF fit params

- Since I cannot reconstruct the network params for di Volo paper fit, I have to use my own fit!

### Di Volo (with adaptation)

In [ ]:
divolo_mft_tf_params = {
    "transfer_function": {
        "fit_transfer_function": True,
        "tf_model": {
            "model_name": "divolo2019",
            "square_terms": True,
        },
        "expansion_point": {
            "voltage_mean": -60.0,
            "voltage_std": 4.0,
            "voltage_tau": 0.5
        },
        "expansion_norm": {
            "voltage_mean": 10.0,
            "voltage_std": 6.0,
            "voltage_tau": 1.0
        },
        "fit_with_w": True,
        "out_rate_min": 0.0,
        "out_rate_max": 200.0,
        "V_eff_fitting": {
            "method": "SLSQP",
            "options": {
                "disp": False,
                "ftol": 1e-15,
                "maxiter": 40000,
            }
        },
        "TF_fitting": {
            "method": "nelder-mead",
            "options": {
                "xtol": 1e-5,
                "disp": False,
                "maxiter": 50000
            },
        }
    }
}

custom_mft_tf_params = {
    "transfer_function": {
        "fit_transfer_function": True,
        "tf_model": {
            "model_name": "neuropsi.custom",
            "square_terms": True,
            "log_term": False,
            "adaptation": True,
        },
        "expansion_point": {
            "voltage_mean": -60.0,
            "voltage_std": 4.0,
            "voltage_tau": 0.5
        },
        "expansion_norm": {
            "voltage_mean": 10.0,
            "voltage_std": 6.0,
            "voltage_tau": 1.0
        },
        "fit_with_w": True,
        "out_rate_min": 0.0,
        "out_rate_max": 200.0,
        "V_eff_fitting": {
            "method": "SLSQP",
            "options": {
                "disp": False,
                'ftol': 1e-9,  # tolerance has to be 1e6 times higher because of units in Zerlaut and custom
                'maxiter':40000,
            }
        },
        "TF_fitting": {
            "method": "nelder-mead",
            "options": {
                "disp": False,
                "xtol": 1e-5,
                "maxiter": 50000, 
            },
        }
    }
}

divolo_mft_tf_params = TestTransferFunctionParams(**divolo_mft_tf_params).transfer_function
custom_mft_tf_params = TestTransferFunctionParams(**custom_mft_tf_params).transfer_function



In [ ]:
# zerlaut TF fit repo, zerlaut TF fit script, zerlaut TF fit MFT, neuropsi_tf MFT

# NOTE: When you want to prove two implementations work the exact same way
# regardless of units, you don't test the optimizer—you test the mathematical
# surface. Optimizers are chaotic algorithms that navigate the surface; 
# if we want to prove the models are identical, we just need to prove that they 
# calculate the exact same error (MSE) for a fixed set of parameters.

tf_fit_results = {}
for neuron_name, neuron_type in zip(["exc_neuron", "inh_neuron"], ["RS-cell", "FS-cell"]):
    divolo_repo_fit = np.load(project_path / "modeldb_divolo2019" / "data" / f"{neuron_type}_CONFIG1_fit.npy", allow_pickle=True)
    tf_fit_results[neuron_name] = {
        "divolo_repo_fit": divolo_repo_fit*1e3,
    }

neuron_results = pynn_mft_results

divolo_mft_tf = tf.run_tf_fitting_workflow(divolo_mft_tf_params, zerlaut_network_params, neuron_results)
neuropsi_mft_tf = tf.run_tf_fitting_workflow(custom_mft_tf_params, zerlaut_network_params, neuron_results)

for neuron_name in ["exc_neuron", "inh_neuron"]:
    tf_fit_results[neuron_name]["divolo_mft_fit"] = np.array(list(divolo_mft_tf[neuron_name].fitted_params.values()))
    tf_fit_results[neuron_name]["neuropsi_mft_fit"] = np.array(list(neuropsi_mft_tf[neuron_name].fitted_params.values()))

In [ ]:
for neuron_name, results in tf_fit_results.items():
    print("\n")
    print(f"Results for {neuron_name}:")
    print(f"{'Parameter':11} | {'Repo':10} | {'Divolo MFT':10} | {'Custom MFT':10}")
    print(f"{'-'*11} | {'-'*10} | {'-'*10} | {'-'*10}")
    for a,b,c, key in zip(*results.values(), list(neuropsi_mft_tf[neuron_name].fitted_params.keys())):
        print(f"{key:11} | {a:10.5f} | {b:10.5f} | {c:10.5f} ")


In [ ]:
# Although different methods give different fitted parameters, they should all
# give the same MSE when we calculate the output rate with those parameters and
# compare to the original output rate from the neuron simulation. This is because
# the mathematical surface of the error should be the same for all implementations, 
# even if the optimizers navigate it differently and end up in different local minima.

for neuron_name, results in tf_fit_results.items():
    print("\n")
    print(f"Results for {neuron_name}:")
    print(f"{'Fit Source':18} | {'DiVolo Script':15} | {'DiVolo MFT MSE':15} | {'Custom MFT MSE':10}")
    print(f"{'-'*18} | {'-'*15} | {'-'*15} | {'-'*10}")

    inh_rate = neuron_results[neuron_name].inh_rate_grid()
    exc_rate = neuron_results[neuron_name].exc_rate_grid()
    out_rate = neuron_results[neuron_name].out_rate_mean()
    adaptation_si = neuron_results[neuron_name].adaptation_mean("A")
    adaptation = neuron_results[neuron_name].adaptation_mean()

    params = zerlaut_repo_results[neuron_name].neuron_params
    Qe = params["Qe"]
    Te = params["Te"]
    Ee = params["Ee"]
    Qi = params["Qi"]
    Ti = params["Ti"]
    Ei = params["Ei"]
    Gl = params["Gl"]
    Cm = params["Cm"]
    El = params["El"]
    Ntot = params["Ntot"]
    pconnec = params["pconnec"]
    gei = params["gei"]

    for fit_source, fit_params in results.items():

        out_rate_script = divolo_tf.TF_my_templateup(exc_rate, inh_rate, adaptation_si, Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, *(fit_params*1e-3))
        mse_script = np.mean((out_rate - out_rate_script)**2)

        fit_params = dict(zip(["P_0", "P_mean", "P_std", "P_tau", "P_log", "P_mean_mean", "P_std_std", "P_tau_tau", "P_mean_std", "P_mean_tau", "P_std_tau"], fit_params))

        divolo_mft = divolo_tf.DiVolo2019TF(neuron_name, zerlaut_network_params, zerlaut_mft_tf_params)

        divolo_mft.set_fitted_parameters(fit_params)

        out_rate_divolo_mft = divolo_mft(exc_rate=exc_rate, inh_rate=inh_rate, adaptation=adaptation)
        mse_divolo_mft = np.mean((out_rate - out_rate_divolo_mft)**2)
        
        custom_mft = neuropsi_tf.NeuroPSICustomTF(neuron_name, zerlaut_network_params, custom_mft_tf_params)
        custom_mft.set_fitted_parameters(fit_params)
        out_rate_custom_mft = custom_mft(exc_rate=exc_rate, inh_rate=inh_rate, adaptation=adaptation)
        mse_custom_mft = np.mean((out_rate - out_rate_custom_mft)**2)
        print(f"{fit_source:18} | {mse_script:15.5f} | {mse_divolo_mft :15.5f} | {mse_custom_mft :10.5f}")

In [ ]:
# membrane fluctuation parameters
# Zerlaut script, di Volo script, MFT custom MembranePotentialFluctuations

step=1

fig, ax = plt.subplots(nrows=5, ncols=2, figsize=(16,16))
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
for i, neuron_name in enumerate(["exc_neuron", "inh_neuron"]):
    for inh_idx in range(neuron_results[neuron_name].inh_rate_grid().shape[1]):
        if inh_idx % step != 0:
            continue
        inh_rate = neuron_results[neuron_name].inh_rate_grid()[:, inh_idx]
        exc_rate = neuron_results[neuron_name].exc_rate_grid()[:, inh_idx]
        out_rate = neuron_results[neuron_name].out_rate_mean()[:, inh_idx]
        adaptation_si = neuron_results[neuron_name].adaptation_mean("A")[:, inh_idx]
        adaptation = neuron_results[neuron_name].adaptation_mean()[:, inh_idx]
        params = zerlaut_repo_results[neuron_name].neuron_params
        Qe = params["Qe"]
        Te = params["Te"]
        Ee = params["Ee"]
        Qi = params["Qi"]
        Ti = params["Ti"]
        Ei = params["Ei"]
        Gl = params["Gl"]
        Cm = params["Cm"]
        El = params["El"]
        Ntot = params["Ntot"]
        pconnec = params["pconnec"]
        gei = params["gei"]

        # cycle through default matplotlib colors
        color = colors[inh_idx % len(colors)]

        divolo_mpf= divolo_tf.get_fluct_regime_varsup(exc_rate, inh_rate, adaptation_si, Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, *([0]*11))
        custom_mpf = neuropsi_tf.MembranePotentialFluctuations(neuron_name=neuron_name, network_params=zerlaut_network_params).evaluate(exc_rate, inh_rate, adaptation=adaptation)
        assert np.allclose(divolo_mpf[0]*1e3, custom_mpf[0], atol=1e-13, rtol=0)
        assert np.allclose(divolo_mpf[1]*1e3, custom_mpf[1], atol=1e-14, rtol=0)
        assert np.allclose(divolo_mpf[2], custom_mpf[4]/zerlaut_network_params.neurons[neuron_name].neuron_params.g_L, atol=1e-14, rtol=0)
        assert np.allclose(divolo_mpf[3], custom_mpf[3], atol=1e-15, rtol=0)


        ax[0,i].plot(exc_rate, divolo_mpf[0]*1e3, 'x', color=color, label=fr"$\nu_{{inh}}={inh_rate[0]:.1f} Hz$")
        ax[0,i].plot(exc_rate, custom_mpf[0], '+', color=color)
        # print(f"Total difference in muV for inh={inh_input:.1f} Hz : ", np.mean(np.abs(muV*1e3 - mu_V)), "mV")

        ax[1,i].plot(exc_rate, divolo_mpf[1]*1e3, 'x', color=color, label=fr"$\nu_{{inh}}={inh_rate[0]:.1f} Hz$")
        ax[1,i].plot(exc_rate, custom_mpf[1], '+', color=color)
        # print(f"Total difference in sigmaV for inh={inh_input:.1f} Hz : ", np.mean(np.abs(sV*1e3 - sigma_V)), "mV")

        ax[2,i].plot(exc_rate, divolo_mpf[3], 'x', color=color, label=fr"$\nu_{{inh}}={inh_rate[0]:.1f} Hz$")
        ax[2,i].plot(exc_rate, custom_mpf[3], '+', color=color)
        # print(f"Total difference in tauVN for inh={inh_input:.1f} Hz : ", np.mean(np.abs(TvN - tau_VN)), "ms")
        
        ax[3,i].plot(exc_rate, divolo_mpf[2], 'x', color=color, label=fr"$\nu_{{inh}}={inh_rate[0]:.1f} Hz$")
        ax[3,i].plot(exc_rate, custom_mpf[4]/zerlaut_network_params.neurons[neuron_name].neuron_params.g_L, '+', color=color)
        # print(f"Total difference in muG for inh={inh_input:.1f} Hz : ", np.mean(np.abs(muGn*1e9 - mu_G/Gl)), "nS")

        mask = out_rate > 0

        divolo_v_eff = divolo_tf.effective_Vthre(out_rate, divolo_mpf[0], divolo_mpf[1], divolo_mpf[3], Gl, Cm)

        # TODO: change to tf results from above
        custom_tf = neuropsi_tf.NeuroPSICustomTF(neuron_name, zerlaut_network_params, custom_mft_tf_params)
        custom_v_eff = custom_tf._get_target_v_eff(out_rate, *custom_mpf[:3])

        assert np.allclose(divolo_v_eff[mask]*1e3, custom_v_eff[mask], atol=1e-13, rtol=0)

        ax[4,i].plot(exc_rate[mask], divolo_v_eff[mask]*1e3, 'x', color=color, label=fr"$\nu_{{inh}}={inh_rate[0]:.1f} Hz$")
        ax[4,i].plot(exc_rate[mask], custom_v_eff[mask], '+', color=color)
        # print(f"Total difference in Vthre_eff for inh={inh_input:.1f} Hz : ", np.mean(np.abs(Vthre_eff*1e3 - v_eff)), "mV")

    ax[0,i].set_title(f"{neuron_name.replace('_', ' ').capitalize()}")
    ax[0,i].set_ylabel(r"$\mu_V$ (mV)")
    ax[1,i].set_ylabel(r"$\sigma_V$ (mV)")
    ax[2,i].set_ylabel(r"$\tau_V^N$ (ms)")
    ax[3,i].set_ylabel(r"$\mu_{G}/G_l$ (nS)")
    ax[4,i].set_ylabel(r"$V_{thre}^{eff}$ (mV)")
    ax[-1,i].set_xlabel(r"$\nu_{exc}$ (Hz)")
fig.tight_layout()
# plt.legend()

# Part IV
I can try few times play with network params if I could find reasonable close params to the transfer function

In [ ]:
zerlaut_repo_results = {}
for neuron_name, cell_type in zip(network_params.internal_neurons, ["RS-cell", "FS-cell"]):
    zerlaut_data = np.load(project_path / "zerlaut2018_data" / f"{cell_type}_CONFIG1.npy", allow_pickle=True)
    
    zerlaut_repo_results[neuron_name] = SingleNeuronResults(
        neuron_name=cell_type,
        neuron_params=zerlaut_data[4],

        exc_rate_grid=zerlaut_data[2].T,
        inh_rate_grid=np.stack([zerlaut_data[3]]*10, axis=1).T,
        out_rate_mean=zerlaut_data[0].T,
        out_rate_std=zerlaut_data[1].T)


zerlaut_network_params = {
    "neurons":{
        "exc_neuron": {
            "neuron_type" : "excitatory",
            "neuron_model" : "adex",
            "neuron_params": {
                "v_rest": zerlaut_repo_results["exc_neuron"].neuron_params["El"]*1e3,
                "v_reset": zerlaut_repo_results["exc_neuron"].neuron_params["Vreset"]*1e3,
                "tau_refrac": zerlaut_repo_results["exc_neuron"].neuron_params["Trefrac"]*1e3,
                "tau_m": zerlaut_repo_results["exc_neuron"].neuron_params["Cm"]/zerlaut_repo_results["exc_neuron"].neuron_params["Gl"]*1e3,
                "cm": zerlaut_repo_results["exc_neuron"].neuron_params["Cm"]*1e9,
                # "cm": 0.22,
                "e_rev_E": zerlaut_repo_results["exc_neuron"].neuron_params["Ee"]*1e3,
                "e_rev_I": zerlaut_repo_results["exc_neuron"].neuron_params["Ei"]*1e3,
                "tau_syn_E": zerlaut_repo_results["exc_neuron"].neuron_params["Te"]*1e3,
                "tau_syn_I": zerlaut_repo_results["exc_neuron"].neuron_params["Ti"]*1e3,
                "a": zerlaut_repo_results["exc_neuron"].neuron_params["a"]*1e9,
                "b": zerlaut_repo_results["exc_neuron"].neuron_params["b"]*1e9,
                # "b": 0.2,
                "delta_T": zerlaut_repo_results["exc_neuron"].neuron_params["delta_v"]*1e3,
                "tau_w": zerlaut_repo_results["exc_neuron"].neuron_params["tauw"]*1e3,
                "v_thresh": zerlaut_repo_results["exc_neuron"].neuron_params["Vthre"]*1e3,
            }
        },
        "inh_neuron": {
            "neuron_type" : "inhibitory",
            "neuron_model" : "adex",
            "neuron_params": {
                "v_rest": zerlaut_repo_results["inh_neuron"].neuron_params["El"]*1e3,
                "v_reset": zerlaut_repo_results["inh_neuron"].neuron_params["Vreset"]*1e3,
                "tau_refrac": zerlaut_repo_results["inh_neuron"].neuron_params["Trefrac"]*1e3,
                "tau_m": zerlaut_repo_results["inh_neuron"].neuron_params["Cm"]/zerlaut_repo_results["inh_neuron"].neuron_params["Gl"]*1e3,
                "cm": zerlaut_repo_results["inh_neuron"].neuron_params["Cm"]*1e9,
                # "cm": 0.22,
                "e_rev_E": zerlaut_repo_results["inh_neuron"].neuron_params["Ee"]*1e3,
                "e_rev_I": zerlaut_repo_results["inh_neuron"].neuron_params["Ei"]*1e3,
                "tau_syn_E": zerlaut_repo_results["inh_neuron"].neuron_params["Te"]*1e3,
                "tau_syn_I": zerlaut_repo_results["inh_neuron"].neuron_params["Ti"]*1e3,
                "a": zerlaut_repo_results["inh_neuron"].neuron_params["a"]*1e9,
                "b": zerlaut_repo_results["inh_neuron"].neuron_params["b"]*1e9,
                "delta_T": zerlaut_repo_results["inh_neuron"].neuron_params["delta_v"]*1e3,
                "tau_w": zerlaut_repo_results["inh_neuron"].neuron_params["tauw"]*1e3,
                "v_thresh": zerlaut_repo_results["inh_neuron"].neuron_params["Vthre"]*1e3,
            },
        },
        "drive_neuron": {
            "neuron_type" : "excitatory",
            "neuron_model" : "poisson_generator",
        },
        "stim_neuron": {
            "neuron_type" : "excitatory",
            "neuron_model" : "poisson_generator",
        }
    },
    "network": {
        "size": {
            "exc_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["Ntot"]*(1-zerlaut_repo_results["exc_neuron"].neuron_params["gei"]),
            "inh_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["Ntot"]*zerlaut_repo_results["exc_neuron"].neuron_params["gei"],
            "drive_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["Ntot"]*(1-zerlaut_repo_results["exc_neuron"].neuron_params["gei"]),
            "stim_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["Ntot"]*(1-zerlaut_repo_results["exc_neuron"].neuron_params["gei"]),
        },
        "connectivity": {
            "exc_neuron":{
                "exc_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["pconnec"],
                "inh_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["pconnec"],
                "drive_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["pconnec"],
                "stim_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["pconnec"]
            },
            "inh_neuron": {
                "exc_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["pconnec"],
                "inh_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["pconnec"],
                "drive_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["pconnec"],
                "stim_neuron": zerlaut_repo_results["exc_neuron"].neuron_params["pconnec"],

            },
            "drive_neuron": {
                "exc_neuron": 0.0,
                "inh_neuron": 0.0,
                "drive_neuron": 0.0,
                "stim_neuron": 0.0,

            },
            "stim_neuron":{
                "exc_neuron": 0.0,
                "inh_neuron": 0.0,
                "drive_neuron": 0.0,
                "stim_neuron": 0.0,
            }
        }
    },
    "synapses": {
        "exc_neuron": {
            "syn_type": "static_synapse",
            "syn_params": {
                "weight": zerlaut_repo_results["exc_neuron"].neuron_params["Qe"]*1e9,
                "delay": 1.0
            }
        },
        "inh_neuron": {
            "syn_type": "static_synapse",
            "syn_params": {
                "weight": zerlaut_repo_results["exc_neuron"].neuron_params["Qi"]*1e9,
                "delay": 1.0
            }
        },
        "drive_neuron": {
            "syn_type": "static_synapse",
            "syn_params": {
                "weight": zerlaut_repo_results["exc_neuron"].neuron_params["Qe"]*1e9,
                "delay": 1.0
            }

        },
        "stim_neuron": {
            "syn_type": "static_synapse",
            "syn_params": {
                "weight": zerlaut_repo_results["exc_neuron"].neuron_params["Qe"]*1e9,
                "delay": 1.0
            }

        }
    }
}

zerlaut_network_params = BiologicalParameters(**zerlaut_network_params)

In [ ]:
pynn_workflow_params = {
    "neuron_simulation": {
        "execution_mode": "run",
        "simulator": "pynn.nest",
        "grid": {
            "exc_neuron" : {
                "grid_type": "custom",
                "exc_rate_grid": zerlaut_repo_results["exc_neuron"].exc_rate_grid(),
                "inh_rate_grid": zerlaut_repo_results["exc_neuron"].inh_rate_grid(),
            },
            "inh_neuron" : {
                "grid_type": "custom",
                "exc_rate_grid": zerlaut_repo_results["inh_neuron"].exc_rate_grid(),
                "inh_rate_grid": zerlaut_repo_results["inh_neuron"].inh_rate_grid(),
            }
        },
        "simulation_time": 10000.0,
        "averaging_window": 10000.0,
        "time_step": 0.05,
        "seed": 4,
        "n_runs": 5,
        "cpus": 16
    }
}

pynn_config = TestNeuronSimulationParams(**pynn_workflow_params).neuron_simulation

print("Config loaded successfully!")
print(f"Running for {pynn_config.simulation_time} ms")

pynn_mft_results = ns.run_neuron_simulation_workflow(pynn_config, zerlaut_network_params)

In [ ]:
divolo_tf_params = {
    "transfer_function": {
        "fit_transfer_function": False,
        "tf_model": {
            "model_name": "divolo2019",
            "square_terms": True,
        },
        "expansion_point": {
            "voltage_mean": -60.0,
            "voltage_std": 4.0,
            "voltage_tau": 0.5
        },
        "expansion_norm": {
            "voltage_mean": 10.0,
            "voltage_std": 6.0,
            "voltage_tau": 1.0
        },
        "tf_fits": None,
    }
}


paper_fit_params = {
    "exc_neuron" : {
        # Di Volo 2019 paper
        # "exc_neuron" : [-49.8, 5.06, -25.0, 1.4, -0.41, 10.5, -36.0, 7.4, 1.2, -40.7],
        "P_0": -49.8,
        "P_mean": 5.06,
        "P_std": -25.0,
        "P_tau": 1.4,
        "P_log": 0.0,
        "P_mean_mean": -0.41,
        "P_mean_std": 7.4,
        "P_mean_tau": 1.2,
        "P_std_std": 10.5,
        "P_std_tau": -40.7,
        "P_tau_tau": -36.0,
    },
    "inh_neuron" : {
        # Di Volo 2019 paper
        # "inh_neuron" : [-51.4, 4.0, -8.3, 0.2, -0.5, 1.4, -14.6, 4.5, 2.8, -15.3],    
        "P_0": -51.4,
        "P_mean": 4.0,
        "P_std": -8.3,
        "P_tau": 0.2,
        "P_log": 0.0,
        "P_mean_mean": -0.5,
        "P_mean_std": 4.5,
        "P_mean_tau": 2.8,
        "P_std_std": 1.4,
        "P_std_tau": -15.3,
        "P_tau_tau": -14.6,
    }
}

repo_fit_params = {
    "exc_neuron": {
        # Di Volo repo
        # "exc_neuron" : [-49.83106, 5.06355, -23.47012, 2.29515, -0.41053, 10.54705, -36.59253, 7.43749, 1.26506, -40.72161],
        "P_0": -49.83106,
        "P_mean": 5.06355,
        "P_std": -23.47012,
        "P_tau": 2.29515,
        "P_log": 0.0,
        "P_mean_mean": -0.41053,
        "P_mean_std": 7.43749,
        "P_mean_tau": 1.26506,
        "P_std_std": 10.54705,
        "P_std_tau": -40.72161,
        "P_tau_tau": -36.59253
    },
    "inh_neuron": {
        # Di Volo repo
        # "inh_neuron" : [-51.49122, 4.00369, -8.35201, 0.24142, -0.50706, 1.43454, -14.68669, 4.50271, 2.84722, -15.3578],    
        "P_0": -51.49122,
        "P_mean": 4.00369,
        "P_std": -8.35201,
        "P_tau": 0.24142,
        "P_log": 0.0,
        "P_mean_mean": -0.50706,
        "P_mean_std": 4.50271,
        "P_mean_tau": 2.84722,
        "P_std_std": 1.43454,
        "P_std_tau": -15.3578,
        "P_tau_tau": -14.68669
    }
}


custom_tf_params = {
    "transfer_function": {
        "fit_transfer_function": True,
        "tf_model": {
            # "model_name": "neuropsi.custom",
            "model_name": "divolo2019",
            "square_terms": True,
            # "log_term": False,
            # "adaptation": True
        },
        "expansion_point": {
            "voltage_mean": -60.0,
            "voltage_std": 4.0,
            "voltage_tau": 0.5
        },
        "expansion_norm": {
            "voltage_mean": 10.0,
            "voltage_std": 6.0,
            "voltage_tau": 1.0
        },
        "fit_with_w": True,
        "out_rate_min": 0.0,
        "out_rate_max": 200.0,
        "V_eff_fitting": {
            "method": "SLSQP",
            "options": {
                "ftol": 5e-9,
                "disp": False,
                "maxiter": 10000
            }
        },
        "TF_fitting": {
            "method": "nelder-mead",
            "options": {
                "fatol": 5e-9,
                "disp": False,
                "maxiter": 10000
            }
        }
    }
}

In [ ]:
divolo_tf_params["transfer_function"]["tf_fits"] = paper_fit_params
# divolo_tf_params["transfer_function"]["tf_fits"] = repo_fit_params

tf_divolo2019_params = TestTransferFunctionParams(**divolo_tf_params).transfer_function
tf_custom_params = TestTransferFunctionParams(**custom_tf_params).transfer_function


tf_funcs_legacy = {
    "exc_neuron": [],
    "inh_neuron": []
}
for tf_params in [tf_divolo2019_params, tf_custom_params]:
    tf_funcs = tf.run_tf_fitting_workflow(tf_params, network_params, pynn_mft_results)
    for neuron_type in tf_funcs.keys():
        tf_funcs_legacy[neuron_type].append(tf_funcs[neuron_type])


In [ ]:
reload(fig_plots)

In [ ]:


fig_name = f"DiVolo_search_for_params.png"
fig_path = project_path / "imgs" / fig_name
fig_plots.fig_tf_fits_together(
    pynn_mft_results, 
    tf_funcs_legacy, 
    # {
    #     "exc_neuron" : [],
    #     "inh_neuron" : []
    # },
    # {
    #     "exc_neuron" : [tf_funcs_legacy["exc_neuron"][0]],
    #     "inh_neuron" : [tf_funcs_legacy["inh_neuron"][0]]
    # },
    common_params={
        'labels' : ["DiVolo paper", "MFT fit"],
        'linestyles' : ['--', ":"],
        # 'xlim' : (0,7),
        'ylim' : (0, 25),
        'linewidth': 2.5,
        'curves_num' : 8,
        "legend": {
            'loc': 'upper left',
        },
    }, 
    fig_params={
        'fontsize': 18,
        'figsize': (15, 10),  # width, height
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
        'title': f"Neuron Activity"
    })
display(Image(filename=str(fig_path)))

In [ ]:
neuron_results_list = [zerlaut_repo_results, zerlaut_mft_results, pynn_mft_results]
plt.rcParams['font.size'] = 18
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

result_labels = ["Zerlaut repo data", "Zerlaut MFT data", "PyNN MFT data"]

idx = [0,1,3,4,5,6,8,9]

for label, neuron_results, marker in zip(result_labels, neuron_results_list, ['o', 'x', '+']):
    axes[0].set_prop_cycle(None)
    axes[1].set_prop_cycle(None)    
    lines_exc = axes[0].plot(neuron_results['exc_neuron'].exc_rate_grid()[:, idx], 
                 neuron_results['exc_neuron'].out_rate_mean()[:, idx], 
                 ls='',
                 marker=marker)
    
    lines_inh = axes[1].plot(neuron_results['inh_neuron'].exc_rate_grid()[:, idx], 
                 neuron_results['inh_neuron'].out_rate_mean()[:, idx], 
                 ls='',
                 marker=marker)
    lines_exc[0].set_label(label)
    lines_inh[0].set_label(label)
axes[0].set_title("exc_neuron")
axes[1].set_title("inh_neuron")
axes[0].set_xlabel(r"$\nu_{e}$ [Hz]")
axes[1].set_xlabel(r"$\nu_{e}$ [Hz]")
axes[0].set_ylabel(r"$\nu_{out}$ [Hz]")
axes[1].set_ylabel(r"$\nu_{out}$ [Hz]")
axes[0].set_ylim(0, 25)
axes[1].set_ylim(0, 25)
axes[0].legend(loc='upper left')
axes[1].legend(loc='upper left')
fig.suptitle("Comparison of Neuron Activity Across Simulators")
fig.tight_layout()